In [ ]:
import sys
import os
sys.path.append('../')
import pathlib as pl
from SymEigen import *
from sympy import symbols
from project_dir import backend_source_dir

Gen = EigenFunctionGenerator()
Gen.MacroBeforeFunction("__host__ __device__")


In [ ]:
D, dHat, kappa, xi = symbols('D dHat kappa xi')
Cl = Gen.Closure(kappa, D, dHat, xi)
# classic log barrier with thickness (codim-shell form)
kB = - kappa * (D - xi * xi - 2 * xi * dHat - dHat * dHat) **2 * log((D-xi * xi)/(2 * xi * dHat + dHat * dHat))
# Stiff-GIPC-style stiff log^2 barrier, zero-thickness (volumetric) form
Cl2 = Gen.Closure(kappa, D, dHat)
kB2 = kappa * (D - dHat * dHat) ** 2 * log(D / (dHat * dHat)) ** 2
# stiff log^2 barrier on the thickness-shifted distance (D - xi^2):
# same active domain (xi^2, (dHat+xi)^2) as the classic form
kB3 = kappa * (D - xi * xi - 2 * xi * dHat - dHat * dHat) ** 2 * log((D - xi * xi) / (2 * xi * dHat + dHat * dHat)) ** 2
kB, kB2, kB3


In [ ]:
dkBdd = kB.diff(D)
dkB2dd = kB2.diff(D)
dkB3dd = kB3.diff(D)
dkBdd, dkB2dd, dkB3dd


In [ ]:
ddkBddd = dkBdd.diff(D)
ddkB2ddd = dkB2dd.diff(D)
ddkB3ddd = dkB3dd.diff(D)
ddkBddd, ddkB2ddd, ddkB3ddd


In [ ]:
s = f'''
// > Squared Version
// > D := d*d

{Cl("KappaBarrierWithThickness",kB)}
{Cl("dKappaBarrierWithThicknessdD",dkBdd)}
{Cl("ddKappaBarrierWithThicknessddD",ddkBddd)}
{Cl2("KappaBarrierLog2",kB2)}
{Cl2("dKappaBarrierLog2dD",dkB2dd)}
{Cl2("ddKappaBarrierLog2ddD",ddkB2ddd)}
{Cl("KappaBarrierLog2WithThickness",kB3)}
{Cl("dKappaBarrierLog2WithThicknessdD",dkB3dd)}
{Cl("ddKappaBarrierLog2WithThicknessddD",ddkB3ddd)}
'''

# Hand-written dispatcher: the stiff log^2 barrier (Stiff-GIPC design) is used for
# both zero-thickness (volumetric) and thickness (codim shell) contacts; the classic
# log barrier with thickness is kept for reference but no longer dispatched.
dispatcher = '''
/* Dispatcher: stiff log^2 barrier (Stiff-GIPC design). xi == 0 -> plain log^2 form;
 * xi > 0 -> log^2 on the thickness-shifted distance (D - xi^2), same active domain
 * (xi^2, (dHat+xi)^2) as the classic codim barrier. The classic log-with-thickness
 * functions (KappaBarrierWithThickness etc.) are kept for reference. */
template <typename T>
__host__ __device__ void KappaBarrier(T& R, const T& kappa, const T& D, const T& dHat, const T& xi)
{
    if(xi == 0.0)
        KappaBarrierLog2(R, kappa, D, dHat);
    else
        KappaBarrierLog2WithThickness(R, kappa, D, dHat, xi);
}
template <typename T>
__host__ __device__ void dKappaBarrierdD(T& R, const T& kappa, const T& D, const T& dHat, const T& xi)
{
    if(xi == 0.0)
        dKappaBarrierLog2dD(R, kappa, D, dHat);
    else
        dKappaBarrierLog2WithThicknessdD(R, kappa, D, dHat, xi);
}
template <typename T>
__host__ __device__ void ddKappaBarrierddD(T& R, const T& kappa, const T& D, const T& dHat, const T& xi)
{
    if(xi == 0.0)
        ddKappaBarrierLog2ddD(R, kappa, D, dHat);
    else
        ddKappaBarrierLog2WithThicknessddD(R, kappa, D, dHat, xi);
}
'''
print(s + dispatcher)

f = open( backend_source_dir('cuda') / 'contact_system/contact_models/sym/codim_ipc_contact.inl', 'w')
f.write(s + dispatcher)
f.close()
